# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar el modelo que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [1]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga del modelo.

In [2]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

In [3]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.3


### **DESCARGAR EL MODELO**

Vamos a trabajar con **Llama 3.1 de 8B**. Guardamos el nombre en la variable `MODELO` para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [4]:
# Descargar el modelo Llama (la primera vez tarda unos minutos)
MODELO = "llama3.1:8b"

!ollama pull {MODELO}
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


### **CLIENTE DE PYTHON**

In [5]:
# Instalar el cliente de Ollama y conectarlo al servidor local
!pip install ollama -q

import ollama

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [6]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.message.content)

Zero-shot: La respuesta es: Positivo.


In [7]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.message.content)
# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada

Few-shot: Mixto.


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [8]:
# Razonamiento paso a paso (chain-of-thought)
problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

¡Vamos a desglosarlo!

**Paso 1: Halla la distancia recorrida por el primer tren en 2 horas**

La distancia recorrida por el primer tren en 2 horas = Velocidad × Tiempo = 80 km/h × 2 horas = 160 km

**Paso 2: Supongamos que el segundo tren tarda x horas en alcanzar al primer tren**

**Paso 3: Calcule la distancia recorrida por el segundo tren en x horas**

La distancia recorrida por el segundo tren en x horas = Velocidad × Tiempo = 120 km/h × x horas = 120x km

**Paso 4: La distancia recorrida por el segundo tren para alcanzar al primero será igual a la distancia recorrida por el primero en 2 horas más la distancia recorrida por el segundo tren en x horas**

120x = 160 + 80x

**Paso 5: Resuelve para x**

80x = 160
x = 2 horas

La respuesta final es 2.


### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [9]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.message.content)
# Por más segura que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación

Lo siento, pero no tengo acceso a información futura o específica sobre eventos como hackathones o su resultado en un momento determinado. Si tienes alguna otra pregunta o necesitas información sobre otro tema, estaré encantado de ayudarte.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [10]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [11]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [12]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [13]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica

Lo sentimos, la política solo menciona "devolución" para tallas, no para productos en oferta o liquidación.


## **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

In [ ]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

In [ ]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

In [14]:
# Descargar el modelo Llama e inicializar el cliente
MODELO = "llama3.1:8b"

!ollama pull {MODELO}
!ollama list

!pip install ollama -q

import ollama

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


**2. Define tu base de conocimiento y genera embeddings:**

* Instala e importa la librería de  `sentence-transformers` junto con `numpy`.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings.

In [ ]:
# Instalar e importar librerías
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [17]:
# Definir la lista documentos y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [20]:
# Definir la función buscar_fragmento
def buscar_fragmento(pregunta, documentos):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    embeddings_documentos = modelo_embeddings.encode(documentos)
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

documentos = ["La colorimetría es la rama de la ciencia que estudia la medición y descripción del color de una manera cuantitativa. Se trata de un conjunto de métodos y estándares que permiten representar, comparar y reproducir colores de forma objetiva y reproducible, tanto en la práctica artística y de diseño como en la industria, la investigación y la tecnología.", "La música es una de las formas de expresión humana más antiguas y universalmente reconocidas. Su capacidad para generar, intensificar, moderar e incluso transformar nuestras emociones la convierte en un fenómeno fascinante que ha sido objeto de estudio en psicología, neurociencia, antropología y arte. A continuación se detalla cómo la música y las emociones interactúan, los mecanismos involucrados y ejemplos concretos de su aplicación práctica.", "El estudio formal de las redes neuronales comenzó en 1943, cuando los neurólogos y matemáticos Warren McCulloch y Walter Pitts publicaron su influyente artículo **“A Logical Calculus of the Ideas Immanent in Nervous Activity”**. En ese trabajo propusieron el primer modelo matemático de un “neurona artificial”, sentando las bases teóricas de lo que hoy conocemos como redes neuronales."]
#pregunta = "¿Cómo se llama al conjunto de métodos y estándares que permiten representar colores?"
pregunta= "¿Existe una relación música-emoción?"
fragmento = buscar_fragmento(pregunta, documentos)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: La música es una de las formas de expresión humana más antiguas y universalmente reconocidas. Su capacidad para generar, intensificar, moderar e incluso transformar nuestras emociones la convierte en un fenómeno fascinante que ha sido objeto de estudio en psicología, neurociencia, antropología y arte. A continuación se detalla cómo la música y las emociones interactúan, los mecanismos involucrados y ejemplos concretos de su aplicación práctica.


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [21]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag
prompt_zero_shot = "¿Existe una relación música-emoción?"

respuesta_sin_rag = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

Zero-shot: La relación entre música y emociones es compleja y multifacética. La música puede evocar emociones en las personas a través de diferentes mecanismos. Por un lado, la música puede estimular respuestas fisiológicas en el cerebro, como la liberación de hormonas como la adrenalina y la dopamina. Estos cambios pueden generar sentimientos de euforia, alegría o incluso ansiedad.

Por otro lado, la música puede ser una forma de expresar y procesar las emociones. Las personas pueden utilizar la música como una vía para abordar experiencias emocionales profundas, como la pérdida de un ser querido o la ansiedad. De esta manera, la música puede actuar como una forma de cúltura y como un refugio emocional.

Además, la música puede influir en la percepción de emociones y en la experiencia de los eventos. Por ejemplo, la música que se escucha mientras se experimenta una situación emocional puede influir en cómo se recuerda y se experimenta esa situación.

En resumen, la relación entre músi

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [22]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag

prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

respuesta_con_rag = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_rag}]
)

print(respuesta_con_rag.message.content)

Sí, existe una relación entre la música y las emociones. La música tiene la capacidad de generar, intensificar, moderar o transformar nuestras emociones.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [23]:
# Mostrar ambas respuestas para comparar
print("respuesta sin RAG:", respuesta_sin_rag.message.content)

respuesta sin RAG: La relación entre música y emociones es compleja y multifacética. La música puede evocar emociones en las personas a través de diferentes mecanismos. Por un lado, la música puede estimular respuestas fisiológicas en el cerebro, como la liberación de hormonas como la adrenalina y la dopamina. Estos cambios pueden generar sentimientos de euforia, alegría o incluso ansiedad.

Por otro lado, la música puede ser una forma de expresar y procesar las emociones. Las personas pueden utilizar la música como una vía para abordar experiencias emocionales profundas, como la pérdida de un ser querido o la ansiedad. De esta manera, la música puede actuar como una forma de cúltura y como un refugio emocional.

Además, la música puede influir en la percepción de emociones y en la experiencia de los eventos. Por ejemplo, la música que se escucha mientras se experimenta una situación emocional puede influir en cómo se recuerda y se experimenta esa situación.

En resumen, la relación en

In [24]:
print("respuesta con RAG:", respuesta_con_rag.message.content)

respuesta con RAG: Sí, existe una relación entre la música y las emociones. La música tiene la capacidad de generar, intensificar, moderar o transformar nuestras emociones.
